# Mixed-Precision Numerics

Training in 16 bits instead of 32 halves memory and roughly doubles throughput on modern
accelerators. It also introduces a class of bug that does not exist in fp32: values that
silently become `inf`, gradients that round to zero, and sums that stop increasing because
the addend is too small to change the accumulator.

None of this is mysterious once you can see the format's actual range and precision. This
notebook makes those visible, then reproduces each failure mode and its standard fix.

Final topic in the [Pre-training](tokenization-bpe.ipynb) track.

## 1. What & Why

A floating-point number is `sign × mantissa × 2^exponent`. How you divide the bits between
mantissa and exponent decides two independent things:

- **Exponent bits → dynamic range.** How large and how small a number you can represent
  at all.
- **Mantissa bits → precision.** How finely you can distinguish numbers that are close
  together.

| Format | Bits | Exponent | Mantissa | Max | Min normal | Decimal digits |
|---|---|---|---|---|---|---|
| **fp32** | 32 | 8 | 23 | ~3.4e38 | ~1.2e-38 | ~7 |
| **fp16** | 16 | 5 | 10 | ~65504 | ~6.1e-5 | ~3 |
| **bf16** | 16 | 8 | 7 | ~3.4e38 | ~1.2e-38 | ~2 |
| **fp8 (E4M3)** | 8 | 4 | 3 | ~448 | ~2e-3 | ~1 |

The crucial comparison is **fp16 versus bf16**. Both are 16 bits, and they make opposite
choices: fp16 spends bits on precision and has a narrow range; bf16 keeps fp32's *entire*
exponent range and sacrifices precision.

For deep learning, **range matters far more than precision** — which is why bf16 has
become the default wherever hardware supports it. Neural network training tolerates noisy
values remarkably well, and tolerates `inf` and `NaN` not at all.

**What "mixed" means:** weights, activations and matmuls in 16-bit; a master copy of the
weights, the optimiser state, and reductions in fp32. You get the speed of the narrow
format and the stability of the wide one.

## 2. Mental Model

**A ruler with a finite length and a finite number of marks.**

- **Range** is the ruler's length. Measure something longer and you get `inf`; shorter
  than its smallest mark and you get `0`. Both are catastrophic and neither is an error
  you will see reported.
- **Precision** is the spacing of the marks. Crucially, floating-point marks are **not
  evenly spaced** — they are dense near zero and sparse far from it. The gap between
  representable numbers near 1.0 is tiny; near 1000 it is much larger.

That non-uniform spacing explains the failure that surprises people most: **adding a small
number to a large one can do nothing at all**. If the addend is smaller than half the gap
between marks at the accumulator's magnitude, the sum rounds back to where it started.
Sum ten thousand small numbers into a 16-bit accumulator and it stalls.

The practical summary:

- fp16 is a **short ruler with fine marks** — precise, and it falls off the end easily.
- bf16 is a **long ruler with coarse marks** — imprecise, and it almost never overflows.

Training cares about not falling off the end.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Dynamic range** | Ratio of largest to smallest representable magnitude. Set by exponent bits. |
| **Machine epsilon** | Smallest `e` with `1 + e ≠ 1`. The relative precision. ~1e-7 (fp32), ~1e-3 (fp16), ~8e-3 (bf16). |
| **Subnormals** | Values below the smallest normal, with reduced precision. Often flushed to zero for speed. |
| **Overflow → `inf`** | Exceeding the max. In fp16 that is 65504 — easily reached by an un-scaled attention score or a squared gradient. |
| **Underflow → `0`** | Below the min. Small gradients vanish in fp16, and a zero gradient is an untrained parameter. |
| **Loss scaling** | Multiply the loss by `S` before backward so gradients land in fp16's range, then divide by `S` before the update. The standard fp16 fix. |
| **Dynamic loss scaling** | Adjust `S` automatically: raise it when steps succeed, halve it and skip the step on `inf`/`NaN`. |
| **Master weights** | An fp32 copy of the weights that the optimiser updates, because a 16-bit update is often too small to register. |
| **Mixed precision** | 16-bit compute, fp32 accumulation and optimiser state. |
| **Stochastic rounding** | Round up or down at random in proportion to distance. Preserves the expected value and defeats accumulator stalling. |
| **Kahan summation** | Track the rounding error in a compensation term and add it back. |

## 4. Setup

NumPy has fp16 and fp32 natively. bf16 is not a NumPy dtype, so we simulate it by
truncating an fp32 mantissa to 7 bits — which is exactly what bf16 is, and makes the
range/precision trade directly measurable.

In [1]:
# %pip install numpy

import numpy as np

def to_bf16(x):
    '''Round fp32 to bfloat16 precision, keeping fp32's exponent range.

    bf16 is the top 16 bits of an fp32 word, so this is a round-to-nearest-even
    truncation of the low 16 bits.
    '''
    x = np.asarray(x, dtype=np.float32)
    u = x.view(np.uint32)
    rounded = (u + 0x7FFF + ((u >> 16) & 1)) & 0xFFFF0000
    return rounded.view(np.float32)

rng = np.random.default_rng(0)
print("numpy", np.__version__)
print("bf16 simulated by mantissa truncation; fp16/fp32 are native")

numpy 2.5.1
bf16 simulated by mantissa truncation; fp16/fp32 are native


## 5. Worked Examples

### Example 1 — the actual limits of each format

Print the numbers rather than trusting the table.

In [2]:
for name, dt in [("fp32", np.float32), ("fp16", np.float16)]:
    info = np.finfo(dt)
    print(f"{name}: max={float(info.max):.3e}  min_normal={float(info.tiny):.3e}  "
          f"eps={float(info.eps):.3e}")

# bf16 keeps fp32's exponent, so its range is fp32's; only precision differs.
one = np.float32(1.0)
eps_bf16 = np.float32(1.0)
while to_bf16(one + eps_bf16 / 2) != one:
    eps_bf16 = eps_bf16 / 2
print(f"bf16: max={float(np.finfo(np.float32).max):.3e}  "
      f"min_normal={float(np.finfo(np.float32).tiny):.3e}  eps~{float(eps_bf16):.3e}")

print("\nThe trade, stated plainly:")
print(f"  fp16 is ~{float(eps_bf16)/float(np.finfo(np.float16).eps):.1f}x MORE precise than bf16")
print(f"  bf16 has ~{float(np.finfo(np.float32).max)/float(np.finfo(np.float16).max):.1e}x "
      f"more range than fp16")

print("\nWhat that means in practice -- can each format hold these values?\n")
print(f"{'value':>12} {'meaning':32} {'fp16':>10} {'bf16':>12}")
cases = [
    (1e-9, "a very small gradient"),
    (1e-8, "a small gradient"),
    (1e-5, "a typical small gradient"),
    (3.0, "a normal activation"),
    (5e4, "a large attention logit"),
    (1e6, "a squared gradient (Adam v)"),
    (1e10, "an exploded value"),
]
for v, meaning in cases:
    with np.errstate(over="ignore"):
        f16 = np.float16(v)
    b16 = to_bf16(np.float32(v))
    f16s = "inf" if np.isinf(f16) else ("0 (underflow)" if f16 == 0 else f"{float(f16):.2e}")
    b16s = "inf" if np.isinf(b16) else ("0 (underflow)" if b16 == 0 else f"{float(b16):.2e}")
    print(f"{v:12.0e} {meaning:32} {f16s:>10} {b16s:>12}")

print("\nfp16 fails at BOTH ends on values that occur routinely in training. bf16")
print("represents every one of them -- imprecisely, which training tolerates.")

fp32: max=3.403e+38  min_normal=1.175e-38  eps=1.192e-07
fp16: max=6.550e+04  min_normal=6.104e-05  eps=9.766e-04
bf16: max=3.403e+38  min_normal=1.175e-38  eps~7.812e-03

The trade, stated plainly:
  fp16 is ~8.0x MORE precise than bf16
  bf16 has ~5.2e+33x more range than fp16

What that means in practice -- can each format hold these values?

       value meaning                                fp16         bf16
       1e-09 a very small gradient            0 (underflow)     9.97e-10
       1e-08 a small gradient                 0 (underflow)     1.00e-08
       1e-05 a typical small gradient           1.00e-05     1.00e-05
       3e+00 a normal activation                3.00e+00     3.00e+00
       5e+04 a large attention logit            5.00e+04     4.99e+04
       1e+06 a squared gradient (Adam v)             inf     9.99e+05
       1e+10 an exploded value                       inf     1.00e+10

fp16 fails at BOTH ends on values that occur routinely in training. bf16
represents e

### Example 2 — loss scaling, and why fp16 needs it

Gradients are small. In fp16, small means zero. Loss scaling shifts the whole gradient
distribution into representable range before the backward pass.

In [3]:
# A realistic gradient distribution: mostly small, log-normally spread.
grads = np.exp(rng.normal(-18, 3.0, 100_000)).astype(np.float32)
print(f"gradient magnitudes: median {np.median(grads):.2e}, "
      f"1st percentile {np.percentile(grads, 1):.2e}\n")

def survival(g, scale, dtype):
    with np.errstate(over="ignore"):        # overflow is what we are measuring
        scaled = (g * scale).astype(dtype)
        lost = float(np.mean(scaled == 0))
        overflowed = float(np.mean(np.isinf(scaled.astype(np.float32))))
    return lost, overflowed

print(f"{'loss scale':>11} {'fp16 zeroed':>12} {'fp16 inf':>10} | {'bf16 zeroed':>12} "
      f"{'bf16 inf':>10}")
for scale in (1, 2**8, 2**12, 2**16, 2**20, 2**24, 2**28, 2**32):
    lz, li = survival(grads, scale, np.float16)
    bz, bi = survival(to_bf16(grads), scale, np.float32)
    print(f"{scale:11d} {lz:12.1%} {li:10.1%} | {bz:12.1%} {bi:10.1%}")

print("\nRead the fp16 columns as a window. Without scaling a large share of the")
print("gradient is silently zeroed -- and a zeroed gradient is not an error, it is a")
print("parameter that quietly stops learning. Scaling recovers it; scaling too far")
print("overflows the other end instead. The usable window is real but bounded.")
print("\nThe usable window is narrow, which is why loss scaling is DYNAMIC in practice:")
print("raise the scale while steps succeed, halve it and skip the step on any inf/NaN.")
print("\nbf16 needs none of this. That is the entire reason it became the default.")

gradient magnitudes: median 1.50e-08, 1st percentile 1.34e-11

 loss scale  fp16 zeroed   fp16 inf |  bf16 zeroed   bf16 inf
          1        58.7%       0.0% |         0.0%       0.0%
        256         5.2%       0.0% |         0.0%       0.0%
       4096         0.6%       0.0% |         0.0%       0.0%
      65536         0.0%       0.0% |         0.0%       0.0%
    1048576         0.0%       0.0% |         0.0%       0.0%
   16777216         0.0%       0.0% |         0.0%       0.0%
  268435456         0.0%       0.1% |         0.0%       0.0%
 4294967296         0.0%       1.1% |         0.0%       0.0%

Read the fp16 columns as a window. Without scaling a large share of the
gradient is silently zeroed -- and a zeroed gradient is not an error, it is a
parameter that quietly stops learning. Scaling recovers it; scaling too far
overflows the other end instead. The usable window is real but bounded.

The usable window is narrow, which is why loss scaling is DYNAMIC in practice:


In [4]:
def dynamic_loss_scale(grad_batches, init=2**15, growth_interval=100):
    '''The standard algorithm, in full. Skip the step on overflow, halve the scale;
    double it after a run of clean steps.'''
    scale, good, skipped, history = init, 0, 0, []
    for g in grad_batches:
        with np.errstate(over="ignore"):
            scaled = (g * scale).astype(np.float16)
            bad = np.any(np.isinf(scaled.astype(np.float32))) or np.any(np.isnan(scaled))
        if bad:
            scale = max(1.0, scale / 2)      # overflow: skip this step, back off
            good, skipped = 0, skipped + 1
        else:
            good += 1
            if good >= growth_interval:      # steady progress: try a larger scale
                scale, good = scale * 2, 0
        history.append(scale)
    return history, skipped

batches = [np.exp(rng.normal(-14, 2.5, 500)).astype(np.float32) for _ in range(600)]
batches[300] = batches[300] * 1e4            # a loss spike partway through
hist, skipped = dynamic_loss_scale(batches)
print(f"steps: {len(batches)}, skipped due to overflow: {skipped}")
print(f"scale at step   0: 2^{np.log2(hist[0]):.0f}")
print(f"scale at step 299: 2^{np.log2(hist[299]):.0f}")
print(f"scale at step 301: 2^{np.log2(hist[301]):.0f}   <- backed off after the spike")
print(f"scale at step 599: 2^{np.log2(hist[-1]):.0f}   <- recovered")
print("\nThe scale tracks the gradient distribution automatically. Skipped steps are")
print("the cost, and a handful out of hundreds is normal and harmless. A run where")
print("MOST steps are skipped means the scale is oscillating -- look at the gradients,")
print("not the scaler.")

steps: 600, skipped due to overflow: 1
scale at step   0: 2^15
scale at step 299: 2^18
scale at step 301: 2^17   <- backed off after the spike
scale at step 599: 2^19   <- recovered

The scale tracks the gradient distribution automatically. Skipped steps are
the cost, and a handful out of hundreds is normal and harmless. A run where
MOST steps are skipped means the scale is oscillating -- look at the gradients,
not the scaler.


### Example 3 — accumulator stalling, and why reductions stay in fp32

Summing many small numbers into a low-precision accumulator does not merely lose
precision — it can stop making progress entirely.

In [5]:
n = 100_000
values = np.full(n, 0.01, dtype=np.float32)
exact = 0.01 * n

def naive_sum(vals, dtype):
    acc = dtype(0)
    for v in vals:
        acc = dtype(acc + dtype(v))
    return float(acc)

def kahan_sum(vals, dtype):
    '''Track the rounding error in a compensation term and feed it back.'''
    acc, comp = dtype(0), dtype(0)
    for v in vals:
        y = dtype(dtype(v) - comp)
        t = dtype(acc + y)
        comp = dtype(dtype(t - acc) - y)
        acc = t
    return float(acc)

sub = values[:20000]
exact_sub = 0.01 * len(sub)
print(f"summing {len(sub):,} copies of 0.01 (exact answer: {exact_sub})\n")
print(f"{'method':28} {'result':>12} {'rel error':>11}")
for name, fn, dt in [("fp16 naive", naive_sum, np.float16),
                     ("fp16 Kahan", kahan_sum, np.float16),
                     ("fp32 naive", naive_sum, np.float32),
                     ("fp32 pairwise (np.sum)", lambda v, d: float(np.sum(v, dtype=d)),
                      np.float32)]:
    r = fn(sub, dt)
    print(f"{name:28} {r:12.2f} {abs(r - exact_sub)/exact_sub:11.2%}")

print("\nThe fp16 naive sum STOPS. Once the accumulator reaches a few hundred, 0.01 is")
print("smaller than half the gap between representable fp16 values there, so every")
print("subsequent addition rounds straight back. It is not drifting -- it is stuck.\n")

acc = np.float16(0)
stalled_at = None
for i, v in enumerate(sub):
    new = np.float16(acc + np.float16(v))
    if new == acc and stalled_at is None:
        stalled_at = (i, float(acc))
        break
    acc = new
print(f"first addition with NO effect: element {stalled_at[0]:,}, "
      f"accumulator = {stalled_at[1]}")
print(f"fp16 gap at that magnitude: {float(np.spacing(np.float16(stalled_at[1]))):.4f} "
      f"-- larger than twice the 0.01 we keep trying to add.")
print("\nThis is why every framework accumulates reductions -- sums, means, softmax")
print("denominators, loss terms, gradient norms -- in fp32 even when the inputs are")
print("16-bit. Kahan summation is the alternative when you cannot widen the")
print("accumulator, and stochastic rounding is the hardware-level answer.")

summing 20,000 copies of 0.01 (exact answer: 200.0)

method                             result   rel error
fp16 naive                          32.00      84.00%
fp16 Kahan                         200.12       0.06%
fp32 naive                         199.97       0.02%
fp32 pairwise (np.sum)             200.00       0.00%

The fp16 naive sum STOPS. Once the accumulator reaches a few hundred, 0.01 is
smaller than half the gap between representable fp16 values there, so every
subsequent addition rounds straight back. It is not drifting -- it is stuck.

first addition with NO effect: element 2,798, accumulator = 32.0
fp16 gap at that magnitude: 0.0312 -- larger than twice the 0.01 we keep trying to add.

This is why every framework accumulates reductions -- sums, means, softmax
denominators, loss terms, gradient norms -- in fp32 even when the inputs are
16-bit. Kahan summation is the alternative when you cannot widen the
accumulator, and stochastic rounding is the hardware-level answer.


### Example 4 — where fp16 actually blows up in a transformer

Attention logits are the classic overflow site: a dot product of two `d`-dimensional
vectors grows with `d`, and fp16 tops out at 65504.

In [6]:
print("Well-behaved activations first (unit-variance q and k):\n")
print(f"{'d_head':>7} {'max |q.k| (unscaled)':>21} {'fp16':>8} {'scaled by 1/sqrt(d)':>21} {'fp16':>8}")
for d in (64, 128, 512, 2048, 8192):
    q = rng.normal(0, 1, (200, d)).astype(np.float32)
    k = rng.normal(0, 1, (200, d)).astype(np.float32)
    logits = q @ k.T
    scaled = logits / np.sqrt(d)
    m, ms = float(np.abs(logits).max()), float(np.abs(scaled).max())
    ok = "inf" if np.isinf(np.float16(m)) else "ok"
    oks = "inf" if np.isinf(np.float16(ms)) else "ok"
    print(f"{d:7d} {m:21.1f} {ok:>8} {ms:21.2f} {oks:>8}")

print("\nWith well-behaved activations, nothing overflows -- the dot product only grows")
print("as sqrt(d). So the textbook worry is not, by itself, the problem.\n")
print("Now with OUTLIER CHANNELS, which real transformers reliably develop -- a few")
print("dimensions carrying magnitudes orders larger than the rest:\n")
print(f"{'d_head':>7} {'outlier scale':>14} {'max |q.k|':>13} {'fp16':>8} "
      f"{'scaled':>11} {'fp16':>8}")
for d, out_scale in [(2048, 1), (2048, 20), (2048, 60), (2048, 120), (8192, 120)]:
    q = rng.normal(0, 1, (200, d)).astype(np.float32)
    k = rng.normal(0, 1, (200, d)).astype(np.float32)
    q[:, :4] *= out_scale                     # a handful of huge channels
    k[:, :4] *= out_scale
    logits = q @ k.T
    scaled = logits / np.sqrt(d)
    with np.errstate(over="ignore"):
        m, ms = float(np.abs(logits).max()), float(np.abs(scaled).max())
        ok = "inf" if np.isinf(np.float16(m)) else "ok"
        oks = "inf" if np.isinf(np.float16(ms)) else "ok"
    print(f"{d:7d} {out_scale:14d} {m:13.1f} {ok:>8} {ms:11.1f} {oks:>8}")

print("\nThat is the real fp16 attention hazard: not dimension, but the outlier")
print("channels that emerge during training. The 1/sqrt(d) scaling buys headroom, and")
print("it is why fp16 attention implementations also clamp or compute logits in fp32.\n")

# The softmax itself has an overflow hazard, and the fix is standard.
logits = np.array([80.0, 20.0, 10.0], dtype=np.float32)
with np.errstate(over="ignore"):
    naive = np.exp(logits.astype(np.float16))
stable_in = (logits - logits.max()).astype(np.float16)
stable = np.exp(stable_in)
print("softmax over logits [80, 20, 10]:")
print(f"  naive exp() in fp16   : {naive}   <- overflowed")
print(f"  after subtracting max : {stable}")
print(f"  normalised            : {stable / stable.sum()}")
print("\nSubtracting the max is mathematically a no-op (softmax is shift-invariant) and")
print("numerically essential. Every real implementation does it; this is why.")

Well-behaved activations first (unit-variance q and k):

 d_head  max |q.k| (unscaled)     fp16   scaled by 1/sqrt(d)     fp16
     64                  38.2       ok                  4.78       ok
    128                  51.7       ok                  4.57       ok
    512                  98.6       ok                  4.36       ok
   2048                 188.2       ok                  4.16       ok
   8192                 459.9       ok                  5.08       ok

With well-behaved activations, nothing overflows -- the dot product only grows
as sqrt(d). So the textbook worry is not, by itself, the problem.

Now with OUTLIER CHANNELS, which real transformers reliably develop -- a few
dimensions carrying magnitudes orders larger than the rest:

 d_head  outlier scale     max |q.k|     fp16      scaled     fp16
   2048              1         190.5       ok         4.2       ok
   2048             20        5363.3       ok       118.5       ok
   2048             60       47251.3 

## 6. Gotchas & Pitfalls

- **Using fp16 without loss scaling.** Example 2. Most of your gradient becomes zero, and
  nothing reports an error — the model just learns worse than it should.
- **Reducing in low precision.** Example 3. Sums, means, softmax denominators, gradient
  norms and loss accumulation belong in fp32 regardless of the compute dtype.
- **Updating weights in 16 bits.** A typical update is `lr × grad` ≈ 1e-7 relative to a
  weight of ~1e-2. In bf16 (eps ≈ 8e-3) that update rounds away completely. Keep fp32
  master weights.
- **Keeping optimiser state in 16 bits.** Adam's second moment is a *squared* gradient,
  so it lives at the square of the gradient's magnitude — straight into fp16 underflow.
- **Assuming bf16 is strictly better.** It has ~3 fewer mantissa bits than fp16. Where
  precision genuinely matters and range does not — some inference kernels, small
  well-conditioned reductions — fp16 is the better 16-bit format.
- **Debugging `NaN` from the wrong end.** A `NaN` is usually the *symptom*; the cause is
  an earlier `inf` (overflow) or a `0/0`. Find the first non-finite tensor, not the last.
- **Comparing fp16 and bf16 runs at the same hyperparameters.** They have different noise
  characteristics; a learning rate tuned for one is not necessarily right for the other.
- **Forgetting that `inf - inf = NaN`.** Overflow in attention logits becomes `NaN` after
  the max-subtraction in softmax, which is why the crash surfaces in a different place
  from where the problem occurred.
- **Trusting a green loss curve.** Silent underflow does not spike the loss. It shows up
  as a model that trains slightly worse than it should, forever.

## 7. When to Use vs Alternatives

| Situation | Format |
|---|---|
| Training on Ampere/Hopper or TPU | **bf16** with fp32 master weights — the default, and no loss scaling needed |
| Training on hardware with no bf16 (e.g. V100) | **fp16** + dynamic loss scaling + fp32 master weights |
| Inference, memory-bound, precision-sensitive | **fp16** — the extra mantissa bits are worth having, and range is less of a risk |
| Very large models, aggressive throughput | **fp8** (E4M3 forward, E5M2 backward) with per-tensor scaling — needs careful calibration |
| Anything numerically delicate | **fp32** — reductions, loss, optimiser state, normalisation statistics |
| Post-training weight compression | Integer quantization — see [Quantization](../03-llm-inference-training-optimization/quantization-gptq-awq.ipynb) |

**The honest position.** On modern hardware this is close to a solved problem: use bf16
autocast with fp32 master weights and fp32 reductions, which is what
`torch.autocast` + `GradScaler` (or DeepSpeed/FSDP mixed precision) give you by default.

The reason to understand the details anyway is debugging. When a run diverges at step
40,000, the difference between "the gradients overflowed", "the accumulator stalled" and
"the optimiser state underflowed" is the difference between a five-minute fix and a week.
Every one of those is visible in the examples above, and none of them raises an exception.

## 8. Resources

- [Mixed Precision Training](https://arxiv.org/abs/1710.03740) — Micikevicius et al.; the paper that introduced loss scaling and fp32 master weights.
- [NVIDIA Mixed Precision Training Guide](https://docs.nvidia.com/deeplearning/performance/mixed-precision-training/index.html) — the practical reference, including the dynamic loss-scaling algorithm of Example 2.
- [BFLOAT16: The secret to high performance on Cloud TPUs](https://cloud.google.com/blog/products/ai-machine-learning/bfloat16-the-secret-to-high-performance-on-cloud-tpus) — why range beat precision for deep learning.
- [PyTorch: Automatic Mixed Precision](https://docs.pytorch.org/docs/stable/amp.html) — what autocast promotes to fp32 and what it leaves in 16-bit; the op list is worth reading once.
- [FP8 Formats for Deep Learning](https://arxiv.org/abs/2209.05433) — the E4M3/E5M2 split and the scaling machinery 8-bit training requires.
- [What Every Computer Scientist Should Know About Floating-Point Arithmetic](https://docs.oracle.com/cd/E19957-01/806-3568/ncg_goldberg.html) — Goldberg; the foundation for everything in this notebook, including Kahan summation.
- [Stochastic Rounding: Algorithms and Hardware Support](https://arxiv.org/abs/2001.01709) — the principled fix for the accumulator stalling of Example 3.